In [1]:
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score
import torch
from datasets import Dataset
import evaluate
import pandas as pd
import gc
import numpy as np
import json
import nltk
from math import ceil
from utils.config import CV_DATA
import re
from utils.model_pipelines import extract_skills_from_text

In [2]:
new_dataset_path= 'Model_dataset/real_world_data.csv'
old_dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

#qa model
old_qa_type_model_path= '.temp/model/google-flan-t5-small'
qa_type_model_result= '.temp/model_results/google-flan-t5-small-v2'
qa_type_model= '.temp/model/google-flan-t5-small-v2'

MAX_TOKEN_SIZE= 2048

In [3]:
CV_DATA.keys()

dict_keys(['availability', 'current_ctc', 'education', 'expected_ctc', 'others', 'personal_information', 'skills', 'working_experience'])

### Preprocessing

In [4]:
data_ratio= {
    "old":20,
    "new":80
    }

new_data_df= pd.read_csv(new_dataset_path)
new_data_df= new_data_df[["question", "question_type", "answer"]]
new_data_df["answer"]= new_data_df["answer"].fillna("")
new_data_df.info()

old_data_df= pd.read_csv(old_dataset_path)
old_data_df= old_data_df[["question", "question_type", "answer"]]
old_data_df["answer"]= old_data_df["answer"].fillna("")
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1892 entries, 0 to 1891
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1892 non-null   object
 1   question_type  1892 non-null   object
 2   answer         1892 non-null   object
dtypes: object(3)
memory usage: 44.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
 2   answer         1644 non-null   object
dtypes: object(3)
memory usage: 38.7+ KB


In [5]:
columns_in_new_df= new_data_df["question_type"].unique()
print(f"columns_in_new_df :{columns_in_new_df}")

columns_in_old_df= old_data_df["question_type"].unique()
print(f"columns_in_old_df :{columns_in_old_df}")

columns_in_new_df :['availability' 'personal_information' 'current_ctc' 'education'
 'working_experience' 'expected_ctc' 'others' 'skills']
columns_in_old_df :['current_ctc' 'expected_ctc' 'personal_information' 'education'
 'working_experience' 'skills' 'availability' 'others']


In [6]:
new_data_df.drop_duplicates(inplace= True)
new_data_df.info()

old_data_df.drop_duplicates(inplace= True)
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1870 entries, 0 to 1891
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1870 non-null   object
 1   question_type  1870 non-null   object
 2   answer         1870 non-null   object
dtypes: object(3)
memory usage: 58.4+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 1637 entries, 0 to 1643
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1637 non-null   object
 1   question_type  1637 non-null   object
 2   answer         1637 non-null   object
dtypes: object(3)
memory usage: 51.2+ KB


#### combaing new and old data

In [7]:
new_data_len= len(new_data_df)
total_len= ceil(new_data_len/(data_ratio["new"]/100))
old_data_len= ceil(total_len- new_data_len)
test_data_len= ceil(total_len/10)

print(f"total dataset len: {total_len}")
print(f"length of new data to add: {new_data_len}")
print(f"length of old data to add: {old_data_len}")
print(f"train dataset length: {total_len- test_data_len}")
print(f"test dataset length: {test_data_len}")

total dataset len: 2338
length of new data to add: 1870
length of old data to add: 468
train dataset length: 2104
test dataset length: 234


In [8]:
temp_df= pd.DataFrame()
while True:
    temp_df= old_data_df.sample(old_data_len)
    columns_in_old_df= temp_df["question_type"].unique()

    if set(columns_in_old_df)== set(columns_in_new_df):
        break
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 468 entries, 915 to 246
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       468 non-null    object
 1   question_type  468 non-null    object
 2   answer         468 non-null    object
dtypes: object(3)
memory usage: 14.6+ KB


In [9]:
old_data_df= temp_df
del temp_df

#### Making a test and train dataset manually
- for old dataset
- for new dataset
- combine both at end

In [10]:
temp_df_1= pd.DataFrame()
temp_df_2= pd.DataFrame()
while True:
    temp_df_1= new_data_df.sample(frac=0.1)
    temp_df_2= new_data_df.drop(temp_df_1.index)

    columns_in_temp_df_1= temp_df_1["question_type"].unique()
    columns_in_temp_df_2= temp_df_2["question_type"].unique()
    if (set(columns_in_new_df)== set(columns_in_temp_df_1)) and (set(columns_in_new_df)== set(columns_in_temp_df_2)): 
        break

In [11]:
new_df_train= temp_df_2
new_df_test= temp_df_1

del temp_df_1, temp_df_2

In [12]:
temp_df_1= pd.DataFrame()
temp_df_2= pd.DataFrame()
while True:
    temp_df_1= old_data_df.sample(frac=0.1)
    temp_df_2= old_data_df.drop(temp_df_1.index)

    columns_in_temp_df_1= temp_df_1["question_type"].unique()
    columns_in_temp_df_2= temp_df_2["question_type"].unique()
    if (set(columns_in_new_df)== set(columns_in_temp_df_1)) and (set(columns_in_new_df)== set(columns_in_temp_df_2)): 
        break

In [13]:
old_df_train= temp_df_2
old_df_test= temp_df_1

del temp_df_1, temp_df_2

In [14]:
train_dataset= pd.concat([new_df_train, old_df_train], ignore_index= False)
test_dataset= pd.concat([new_df_test, old_df_test], ignore_index= False)

In [15]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2104 entries, 0 to 246
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       2104 non-null   object
 1   question_type  2104 non-null   object
 2   answer         2104 non-null   object
dtypes: object(3)
memory usage: 65.8+ KB


In [16]:
test_dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 234 entries, 1519 to 277
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       234 non-null    object
 1   question_type  234 non-null    object
 2   answer         234 non-null    object
dtypes: object(3)
memory usage: 7.3+ KB


### mapping content

In [17]:
def map_context(row):
    if row['question_type'] == 'skills':
        text= row['question']
        context= []
        skill_list= extract_skills_from_text(text)
        if not skill_list:
            return CV_DATA["skills"]
        context= [line for line in CV_DATA["skills"].split("\n") if any(re.search(rf'\b{skill}\b', line, re.IGNORECASE) for skill in skill_list)]
        return '\n'.join(context)
    return CV_DATA.get(row['question_type'], None)

train_dataset["context"] = train_dataset.apply(map_context, axis=1)
test_dataset["context"] = test_dataset.apply(map_context, axis=1)

In [18]:
# train_dataset["context"] = train_dataset["question_type"].map(CV_DATA)
# test_dataset["context"] = test_dataset["question_type"].map(CV_DATA)

In [19]:
train_dataset= train_dataset.sample(frac=1).reset_index(drop=True)
test_dataset= test_dataset.sample(frac=1).reset_index(drop=True)
train_dataset.head()

,question,question_type,answer,context
0,Years of experience working on project(s) invo...,working_experience,No,Experience: 2+ years as Software Engineer/ Dat...
1,What is the notice period you have to serve at...,availability,30 days,Interview Availability: 2 PM – 9 PM daily.\n\n...
2,In how many days you join.,availability,30 days,Interview Availability: 2 PM – 9 PM daily.\n\n...
3,How many years of experience do you have with ...,skills,2 years,SQL | Rating: 8/10 | Experience: 2 years | Exp...
4,What are your expectations for yearly bonuses?,expected_ctc,Varies,Expected Compensation (CTC- ₹ 8.5 LPA / $10240...


In [20]:
train_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2104 entries, 0 to 2103
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       2104 non-null   object
 1   question_type  2104 non-null   object
 2   answer         2104 non-null   object
 3   context        2104 non-null   object
dtypes: object(4)
memory usage: 65.9+ KB


### Retraing Preparations:

In [21]:
# Preprocess data
def preprocess_data(row):
    input_text = f"You are a candidate filling job application form answer the question based on the given information.\nQuestion: {row['question']}\nContext: {row['context']}"
    target_text = row['answer']
    return {"input_text": input_text, "target_text": target_text}

processed_data = train_dataset.apply(preprocess_data, axis=1)
train_dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

processed_data = test_dataset.apply(preprocess_data, axis=1)
test_dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

In [22]:
tokenizer = T5Tokenizer.from_pretrained(old_qa_type_model_path)
tokenizer.model_max_length = MAX_TOKEN_SIZE
def tokenize_data(example):
    input_encodings = tokenizer(
        example["input_text"], truncation=True, padding="max_length", max_length=MAX_TOKEN_SIZE
    )
    target_encodings = tokenizer(
        example["target_text"], truncation=True, padding="max_length", max_length=512
    )
    input_encodings["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in labels]
        for labels in target_encodings["input_ids"]
    ]
    return input_encodings



train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

Map:   0%|          | 0/2104 [00:00<?, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

In [23]:
def extend_t5_positional_embeddings(model, new_max_length=2048):
    """ Properly extends T5 model to handle longer sequences """
    model.config.n_positions = new_max_length  # This may be ignored since T5 relies on relative embeddings
    model.config.max_length = new_max_length
    model.resize_token_embeddings(model.config.vocab_size)  # Ensures embedding size matches vocab
    
    return model

# Train

In [24]:
model = T5ForConditionalGeneration.from_pretrained(old_qa_type_model_path)
# model = extend_t5_positional_embeddings(model, new_max_length= MAX_TOKEN_SIZE)
# model.config.max_position_embeddings = MAX_TOKEN_SIZE

In [25]:
model.config.use_cache = False  # Disable caching during training
model.gradient_checkpointing_enable()

In [26]:
for param in model.base_model.parameters():
    param.requires_grad = False  # Freeze lower layers

# Fine-tune only the last few layers
for param in model.decoder.parameters():  # Modify based on model type
    param.requires_grad = True


In [27]:
training_args = TrainingArguments(
    output_dir=qa_type_model_result,  
    eval_strategy="epoch",  # Save based on steps instead of per epoch
    save_strategy="epoch",
    save_steps=500,  # Save model every 500 steps
    learning_rate=5e-5,
    num_train_epochs=10,  # Reduced to prevent overfitting
    per_device_train_batch_size=2,  # Increase if GPU allows 
    per_device_eval_batch_size=2, 
    gradient_accumulation_steps=1,  # Adjust if batch size is low
    logging_dir="./logs",
    logging_steps=50,  
    save_total_limit=3,  # Reduce to save storage space
    warmup_steps=300, 
    weight_decay=0.01,  
    adam_epsilon=1e-8,  
    max_grad_norm=0.5,  # Default is 1.0
    # use_cpu= True,
    dataloader_pin_memory=False,  
    load_best_model_at_end=True,
)


In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)    

torch.cuda.empty_cache()
gc.collect()

trainer.train()

{'loss': 1.0119, 'grad_norm': 6.2548627853393555, 'learning_rate': 8.333333333333334e-06, 'epoch': 0.04752851711026616}
{'loss': 1.1616, 'grad_norm': 20.990251541137695, 'learning_rate': 1.6666666666666667e-05, 'epoch': 0.09505703422053231}
{'loss': 0.8961, 'grad_norm': 3.146772623062134, 'learning_rate': 2.5e-05, 'epoch': 0.14258555133079848}
{'loss': 1.0379, 'grad_norm': 3.7531676292419434, 'learning_rate': 3.3333333333333335e-05, 'epoch': 0.19011406844106463}
{'loss': 0.8157, 'grad_norm': 8.242061614990234, 'learning_rate': 4.166666666666667e-05, 'epoch': 0.2376425855513308}
{'loss': 1.0743, 'grad_norm': 8.860265731811523, 'learning_rate': 5e-05, 'epoch': 0.28517110266159695}
{'loss': 0.9769, 'grad_norm': 9.011494636535645, 'learning_rate': 4.9755381604696675e-05, 'epoch': 0.33269961977186313}
{'loss': 0.8909, 'grad_norm': 5.968137741088867, 'learning_rate': 4.951076320939335e-05, 'epoch': 0.38022813688212925}
{'loss': 0.7617, 'grad_norm': 8.27663516998291, 'learning_rate': 4.926614

TrainOutput(global_step=4208, training_loss=0.7127021489941122, metrics={'train_runtime': 11270.314, 'train_samples_per_second': 1.867, 'train_steps_per_second': 0.933, 'train_loss': 0.7127021489941122, 'epoch': 4.0})

In [29]:
# last_checkpoint = ".temp/model_results/google-flan-t5-small-v2/checkpoint-1052"  # Replace with actual latest checkpoint
# trainer.train(resume_from_checkpoint=last_checkpoint)


In [30]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.6790460348129272, 'eval_runtime': 84.9286, 'eval_samples_per_second': 2.755, 'eval_steps_per_second': 1.378, 'epoch': 4.0}


{'eval_loss': 0.6790460348129272,
 'eval_runtime': 84.9286,
 'eval_samples_per_second': 2.755,
 'eval_steps_per_second': 1.378,
 'epoch': 4.0}

In [31]:
trainer.save_model(qa_type_model)
tokenizer.save_pretrained(qa_type_model)

('.temp/model/google-flan-t5-small-v2/tokenizer_config.json',
 '.temp/model/google-flan-t5-small-v2/special_tokens_map.json',
 '.temp/model/google-flan-t5-small-v2/spiece.model',
 '.temp/model/google-flan-t5-small-v2/added_tokens.json')